In [ ]:
import base64
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding
 
# Load public key
with open("public_key.pem", "rb") as f:
    public_key = serialization.load_pem_public_key(f.read())
 
password = ""
 
encrypted = public_key.encrypt(
    password.encode(),
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
 
encrypted_password = base64.b64encode(encrypted).decode()
 
# print(encrypted_password)

In [ ]:
import os
import requests

BASE = "https://analytics-dev.tatapower.com/TpcDSMPortal/api"

# Headers that make us look like a normal client (defeats WAF 406 blocks)
COMMON_HEADERS = {
    "User-Agent": "PostmanRuntime/7.39.0",   # or a browser UA string
    "Accept": "*/*",
}

# --- Step 1: get an access token ---
def get_token() -> str:
    resp = requests.post(
        f"{BASE}/token",
        headers=COMMON_HEADERS,
        data={  # form-urlencoded, matches Postman's "urlencoded" body
            "grant_type": "password",
            "username": "",
            "password": "",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]  # adjust key if the response differs


# --- Step 2: call the utilityData endpoint ---
def get_utility_data(token: str, plant_name: str, date: str, schd_rev_no: int = -1) -> dict:
    resp = requests.post(
        f"{BASE}/utilityData",
        headers={**COMMON_HEADERS, "Authorization": f"Bearer {token}"},
        json={  # JSON body, matches Postman's "raw"/json body
            "plant_name": plant_name,
            "date": date,
            "schd_rev_no": schd_rev_no,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


# --- Usage ---
# Read credentials from environment variables, NOT hardcoded in the notebook
token = get_token()
data = get_utility_data(token, plant_name="Pavagada", date="2026-06-03", schd_rev_no=-1)
print(data)


In [ ]:
data